# Run Sample Pipeline

Run the `doc-grader` pipeline on a 10 document sample.

In [ ]:
import subprocess
from pathlib import Path

repo_root = Path("..").resolve()

par_config = repo_root / "config" / "profiles" / "ipp_2024_25_par.json"
int_config = repo_root / "config" / "profiles" / "ipp_2024_25_int.json"

eval_nm_config = repo_root / "config" / "experiments" / "evaluation" / "normal_judge_mini_content.json"
eval_mn_config = repo_root / "config" / "experiments" / "evaluation" / "mini_judge_nano_content.json"
eval_mnnl_config = repo_root / "config" / "experiments" / "evaluation" / "mini_judge_nano_content_nolocal.json"

par_out = repo_root / "data" / "judge_gold" / "toolout_sample" / "par"
int_out = repo_root / "data" / "judge_gold" / "toolout_sample" / "int"

eval_nm_out = repo_root / "data" / "judge_gold" / "toolout_sample" / "int_normal_judge_mini_content"
eval_mn_out = repo_root / "data" / "judge_gold" / "toolout_sample" / "int_mini_judge_nano_content"
eval_mnnl_out = repo_root / "data" / "judge_gold" / "toolout_sample" / "int_mini_judge_nano_content_nolocal"

# Uncomment to run on specific students
par_students = [
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT"
]
int_students = [
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT",
    # "REDACTED_STUDENT"
]

def run_pipeline(student_id: str, variant: str, config_path: Path, out_dir: Path):
    target_dir = out_dir / student_id
    if (target_dir / "info.json").exists():
        print(f"\n[{variant.upper()}] Skipping {student_id} (already processed)")
        return

    data_path = repo_root / "data" / "judge_gold" / "int" / student_id if "eval" in variant else repo_root / "data" / "judge_gold" / variant / student_id
    
    print(f"\n[{variant.upper()}] Processing {student_id}...")
    
    cmd = [
        "doc-grader",
        str(data_path),
        "--out", str(out_dir),
        "--config", str(config_path)
    ]
    # Stream output to the notebook execution cell
    subprocess.run(cmd, check=True, cwd=repo_root)

In [ ]:
print("PAR variant")
for sid in par_students:
    run_pipeline(sid, "par", par_config, par_out)

print("\nINT variant")
for sid in int_students:
    run_pipeline(sid, "int", int_config, int_out)

print("\nINT EVAL: normal_judge_mini_content")
for sid in int_students:
    run_pipeline(sid, "eval_nm", eval_nm_config, eval_nm_out)

print("\nINT EVAL: mini_judge_nano_content")
for sid in int_students:
    run_pipeline(sid, "eval_mn", eval_mn_config, eval_mn_out)

print("\nINT EVAL: mini_judge_nano_content_nolocal")
for sid in int_students:
    run_pipeline(sid, "eval_mnnl", eval_mnnl_config, eval_mnnl_out)

print(f"\nDone. Outputs saved to {par_out.relative_to(repo_root)}\n")

In [ ]:
# Get all student IDs from the full INT gold dataset
int_gold_dir = repo_root / "data" / "judge_gold" / "int"
if int_gold_dir.exists():
    all_int_students = [p.name for p in int_gold_dir.iterdir() if p.is_dir()]
    print(f"Found {len(all_int_students)} students in the full INT gold dataset.")
    
    # Define toolout directories for the full run
    full_int_out = repo_root / "data" / "judge_gold" / "toolout_full" / "int"
    full_eval_nm_out = repo_root / "data" / "judge_gold" / "toolout_full" / "int_normal_judge_mini_content"
    full_eval_mn_out = repo_root / "data" / "judge_gold" / "toolout_full" / "int_mini_judge_nano_content"
    full_eval_mnnl_out = repo_root / "data" / "judge_gold" / "toolout_full" / "int_mini_judge_nano_content_nolocal"
    
    # Standard INT
    print("\nRunning standard INT pipeline on full dataset...")
    for sid in all_int_students:
        run_pipeline(sid, "int", int_config, full_int_out)
        
    print("\nRunning normal_judge_mini_content pipeline on full INT dataset...")
    for sid in all_int_students:
        run_pipeline(sid, "eval_nm", eval_nm_config, full_eval_nm_out)
        
    print("\nRunning mini_judge_nano_content pipeline on full INT dataset...")
    for sid in all_int_students:
        run_pipeline(sid, "eval_mn", eval_mn_config, full_eval_mn_out)
        
    print("\nRunning mini_judge_nano_content_nolocal pipeline on full INT dataset...")
    for sid in all_int_students:
        run_pipeline(sid, "eval_mnnl", eval_mnnl_config, full_eval_mnnl_out)
        
    print(f"\nDone. Outputs saved to {full_int_out.parent.relative_to(repo_root)}\n")
else:
    print(f"Directory {int_gold_dir} not found.")